<a href="https://colab.research.google.com/github/elizebeth18/Learn-python/blob/main/Python_6_Exception_Handling_Class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exception Handling in Python


### What you'll learn
1. What errors and exceptions are, and the most common ones you'll meet
2. How to use `try`, `except`, `else`, and `finally`
3. How to signal your own errors with `raise`
4. How to design your own **custom exceptions**


## 1. What *is* an exception?

In Python vocabulary:

- An **error** is a problem in your code.
- An **exception** is Python's way of *raising its hand* to say "something went wrong here!" — it's an object that describes the problem.
- When an exception is *raised* and nobody *catches* it, the program crashes and prints a **traceback** (that red block of text).


## 2. Two kinds of problems: Syntax Errors vs Exceptions

Before we handle exceptions, it's important to know they are **not** the same as syntax errors.

| | **Syntax Error** | **Exception** |
|---|---|---|
| **When it happens** | *Before* the program runs (Python can't even read your code) | *While* the program is running |
| **Cause** | You broke Python's grammar rules (missing `:`, unbalanced brackets) | The code is valid, but something went wrong during execution |
| **Can you catch it?** | ❌ No — you must fix the code | ✅ Yes — this is what exception handling is for |
| **Example** | `print("hi"` (missing bracket) | `10 / 0` (valid code, but math is impossible) |

The cell below has a **syntax error**. It won't run at all — Python rejects it before execution.


In [ ]:
# ⚠️ SYNTAX ERROR DEMO — this cell is MEANT to fail.
# The closing parenthesis is missing, so Python can't understand the line.
# Un-comment the next line to see the SyntaxError, then comment it back.

# print("Hello world"


In [ ]:
# This, by contrast, is perfectly valid Python — no syntax error.
# But it raises an EXCEPTION when it runs, because dividing by zero is impossible.
# Run it to see the traceback (the red error block).

print(10 / 0)   # ZeroDivisionError!


## 3. Common built-in exceptions

| Exception | Raised when… | Tiny example |
|---|---|---|
| `ZeroDivisionError` | You divide a number by zero | `1 / 0` |
| `ValueError` | A function gets the *right type* but an *inappropriate value* | `int("hello")` |
| `TypeError` | An operation is used on the *wrong type* | `"2" + 2` |
| `IndexError` | You access a list index that doesn't exist | `[1, 2][5]` |
| `KeyError` | You access a dictionary key that doesn't exist | `{"a": 1}["b"]` |
| `NameError` | You use a variable that was never defined | `print(xyz)` |
| `AttributeError` | You access a method/attribute an object doesn't have | `"hi".push()` |
| `FileNotFoundError` | You try to open a file that isn't there | `open("nope.txt")` |
| `ImportError` / `ModuleNotFoundError` | An import fails | `import banana` |



In [ ]:
# ValueError: correct type (string) but a value int() can't convert
int("hello")


ValueError: invalid literal for int() with base 10: 'hello'

In [ ]:
# TypeError: you can't add a string and an integer together
"2" + 2


TypeError: can only concatenate str (not "int") to str

In [ ]:
# IndexError: the list has indexes 0 and 1 only; index 5 doesn't exist
numbers = [10, 20]
print(numbers[5])


IndexError: list index out of range

In [ ]:
# KeyError: the dictionary has no key called "phone"
person = {"name": "Ada", "age": 36}
print(person["phone"])


KeyError: 'phone'

### How to read a traceback

When an exception isn't caught, Python prints a **traceback**. Read it **from the bottom up**:

```
Traceback (most recent call last):
  File "example.py", line 3, in <module>
    print(person["phone"])
KeyError: 'phone'
```

- **Bottom line** → the *type* of exception (`KeyError`) and a short message (`'phone'`). Read this first!
- **Middle lines** → *where* it happened (file name, line number, and the offending code).
- **Top line** → "most recent call last" tells you the calls are listed in order, with the deepest (where the error actually occurred) at the bottom.

👉 **Beginner tip:** 90% of debugging is just reading that bottom line carefully.


## 4. Why bother handling exceptions?

Consider a simple program that asks the user for a number and divides 100 by it.

Watch what happens if the user types `0` or a word instead of a number — the program **crashes**, and any code *after* the crash never runs.


In [ ]:
# ⚠️ FRAGILE VERSION — no protection.
# If `user_input` can't become an int, or is 0, this crashes.
# (We hard-code the input here so the notebook doesn't wait for typing.)

user_input = "0"          # pretend the user typed 0
number = int(user_input)  # would raise ValueError if it were "abc"
result = 100 / number     # raises ZeroDivisionError because number is 0

print("Result is:", result)
print("Thanks for using the program!")   # 😢 this line NEVER runs — crash above


## 5. `try` / `except`:

The core tool is the **`try` / `except`** block.

```python
try:
    # code that MIGHT go wrong
except SomeError:
    # what to do IF that error happens
```

- Python runs the code in `try`.
- If **no** exception occurs, the `except` block is **skipped**.
- If an exception occurs, Python **stops the `try` block** and jumps to the matching `except`.
- Either way, the program **keeps running** afterwards.

Here's the fragile program from above, now made safe:


In [ ]:
# ✅ SAFE VERSION — the crash is caught and handled gracefully.

user_input = "0"   # pretend the user typed 0

try:
    number = int(user_input)     # might raise ValueError
    result = 100 / number        # might raise ZeroDivisionError
    print("Result is:", result)
except ZeroDivisionError:
    # This runs ONLY if a ZeroDivisionError happened in the try block.
    print("Oops — you can't divide by zero. Please enter a non-zero number.")

# Because the error was handled, the program continues normally:
print("Thanks for using the program!")   # 🎉 this line DOES run now


Oops — you can't divide by zero. Please enter a non-zero number.
Thanks for using the program!


## 6. Catching *specific* exceptions

The safe version above only handles `ZeroDivisionError`. But what if the user types `"abc"`? That's a **`ValueError`**, which our `except` doesn't cover — so it would still crash.

You can handle **multiple exception types** by stacking `except` blocks. Python checks them **top to bottom** and runs the **first one that matches**.


In [ ]:
def divide_100_by(user_input):
    try:
        number = int(user_input)   # ValueError if not a number
        result = 100 / number      # ZeroDivisionError if number is 0
        print("Result is:", result)
    except ValueError:
        print(f"'{user_input}' is not a whole number. Please enter digits only.")
    except ZeroDivisionError:
        print("You can't divide by zero. Please enter a non-zero number.")

# Try it with three different inputs:
divide_100_by("25")     # works normally  -> Result is: 4.0
divide_100_by("abc")    # triggers ValueError
divide_100_by("0")      # triggers ZeroDivisionError


Result is: 4.0
'abc' is not a whole number. Please enter digits only.
You can't divide by zero. Please enter a non-zero number.


### Handling several exceptions the *same* way

If you want to react to **different exceptions with identical code**, group them in a tuple:

```python
except (ValueError, TypeError):
    print("Something about your input was wrong.")
```


In [ ]:
def to_positive_int(value):
    try:
        n = int(value)
        return abs(n)
    except (ValueError, TypeError):
        # Handles BOTH: int("abc") -> ValueError, int(None) -> TypeError
        print(f"Could not convert {value!r} to a number. Returning 0.")
        return 0

print(to_positive_int("-8"))    # 8
print(to_positive_int("abc"))   # message + 0
print(to_positive_int(None))    # message + 0


8
Could not convert 'abc' to a number. Returning 0.
0
Could not convert None to a number. Returning 0.
0


## 8. The `else` clause: "only if nothing went wrong"

A `try` block can have an optional **`else`**. Its code runs **only if the `try` block succeeded** (no exception raised).

**Why not just put that code inside `try`?** Because `else` makes your intent crystal clear: *"this part is the follow-up work that should only happen on success, and I do NOT want its errors accidentally caught by the except above."*

```
try:     → risky code
except:  → runs if try failed
else:    → runs if try SUCCEEDED
```


In [ ]:
def read_age(text):
    try:
        age = int(text)          # the risky part
    except ValueError:
        print(f"'{text}' isn't a valid age.")
    else:
        # Runs ONLY when int(text) succeeded. Cleaner than piling it into try.
        print(f"Success! Next year you'll be {age + 1}.")

read_age("29")     # else runs -> "Next year you'll be 30."
read_age("old")    # except runs -> error message, else is skipped


Success! Next year you'll be 30.
'old' isn't a valid age.


## 9. The `finally` clause: "always run this, no matter what"

`finally` runs **every single time** — whether the `try` succeeded, raised an exception, or even returned early. It's the perfect place for **cleanup**: closing files, releasing resources, disconnecting from a database.

```
try:     → risky code
except:  → runs if it failed
else:    → runs if it succeeded
finally: → runs ALWAYS (cleanup)
```

Think of `finally` as *"turn off the stove before you leave the kitchen — whether dinner succeeded or burned."* 🔥


In [ ]:
def process(number):
    print(f"\n--- Processing {number} ---")
    try:
        result = 100 / number
        print("Result:", result)
    except ZeroDivisionError:
        print("Can't divide by zero!")
    finally:
        # This ALWAYS prints — success or failure.
        print("Cleanup done. (This runs no matter what.)")

process(4)    # success -> then cleanup
process(0)    # error   -> handled -> then cleanup


### The classic use: closing a file safely

Even if reading a file blows up halfway, `finally` guarantees the file gets closed so we don't leak resources.

> In real code you'd usually prefer a `with open(...)` block, which closes the file for you automatically. But `finally` shows *how* that cleanup guarantee works under the hood.


In [ ]:
# Create a small file first so we have something to read.
with open("demo.txt", "w") as f:
    f.write("line one\nline two\n")

f = None
try:
    f = open("demo.txt", "r")
    print(f.read())
    # Imagine some processing here that might raise an exception...
except FileNotFoundError:
    print("The file was not found.")
finally:
    # Guarantee the file is closed, error or not.
    if f is not None:
        f.close()
        print("File closed safely in finally.")


## 10. Putting `try` / `except` / `else` / `finally` together

Here is the full four-part structure in one example. Read the comments to trace which parts run for each call.


In [ ]:
def withdraw(balance, amount):
    print(f"\nAttempting to withdraw {amount} from balance {balance}")
    try:
        amount = int(amount)                 # might raise ValueError
        new_balance = balance - amount       # simple maths
        if new_balance < 0:
            raise ValueError("Insufficient funds")  # we raise our own (see Sec 11)
    except ValueError as e:
        print(f"❌ Transaction failed: {e}")
        return balance                       # unchanged balance
    else:
        print(f"✅ Success! New balance: {new_balance}")
        return new_balance
    finally:
        print("📝 Transaction attempt logged.")   # always runs

balance = 100
balance = withdraw(balance, 30)      # success
balance = withdraw(balance, "abc")   # ValueError from int()
balance = withdraw(balance, 999)     # our raised "Insufficient funds"



Attempting to withdraw 30 from balance 100
✅ Success! New balance: 70
📝 Transaction attempt logged.

Attempting to withdraw abc from balance 70
❌ Transaction failed: invalid literal for int() with base 10: 'abc'
📝 Transaction attempt logged.

Attempting to withdraw 999 from balance 70
❌ Transaction failed: Insufficient funds
📝 Transaction attempt logged.


## 11. Raising exceptions yourself with `raise` 🚩

So far we've *caught* exceptions Python threw at us. But you can also **throw your own** when your code detects a situation that shouldn't be allowed. This is how you enforce rules.

```python
raise ExceptionType("a helpful message explaining what went wrong")
```

**Why raise deliberately?** It stops bad data from silently travelling deeper into your program. Fail loudly, fail early — with a clear message.


In [ ]:
def set_age(age):
    # Guard clauses that raise on invalid input.
    if not isinstance(age, int):
        raise TypeError("Age must be an integer.")
    if age < 0:
        raise ValueError("Age cannot be negative.")
    if age > 150:
        raise ValueError("Age is unrealistically large.")
    print(f"Age set to {age}.")

set_age(25)      # fine

# Each of these raises — try un-commenting them one at a time:
# set_age(-5)     # ValueError: Age cannot be negative.
# set_age("30")   # TypeError: Age must be an integer.
# set_age(999)    # ValueError: Age is unrealistically large.


Age set to 25.


## 15. Quick reference summary 🗂️

| Keyword | Purpose |
|---|---|
| `try` | Wrap code that might raise an exception |
| `except` | Handle a specific exception if it occurs |
| `except (A, B)` | Handle several exception types the same way |
| `except E as e` | Capture the exception object for its details |
| `else` | Run only if the `try` block **succeeded** |
| `finally` | Run **always** (cleanup), success or failure |
| `raise` | Throw an exception yourself |
| `raise` (bare, in `except`) | Re-raise the current exception |
| `class MyError(Exception)` | Define a custom exception type |

**Golden rules:**
1. Catch **specific** exceptions, not bare `except:`.
2. Keep the `try` block **small** — only the risky line(s).
3. Never **silently swallow** errors; at minimum, log them.
4. Use `finally` for **cleanup** that must always happen.
5. `raise` early with **clear messages** to stop bad data spreading.
6. Create **custom exceptions** for your program's own rules.


## Assignment 1 — Safe division  ⭐ *Beginner (10 pts)*

Write a function `safe_divide(a, b)` that returns `a / b`, but:
- If `b` is `0`, catch the `ZeroDivisionError` and return the string `"Cannot divide by zero"`.

Keep the `try` block small — only the division belongs in it.


In [ ]:
def safe_divide(a, b):
    # YOUR CODE HERE
    try:
      return (a/b)
    except ZeroDivisionError:
      return f"Cannot divide by zero"
    else:
      return (a/b)


# --- Test cases (do not modify) ---
print(safe_divide(10, 2))    # Expected: 5.0
print(safe_divide(9, 3))     # Expected: 3.0
print(safe_divide(5, 0))     # Expected: Cannot divide by zero


5.0
3.0
Cannot divide by zero


## Assignment 2 — Robust integer parser  ⭐⭐ *Easy (15 pts)*

Write a function `parse_int(text)` that:
- Returns the integer value of `text` if it can be converted.
- Returns `None` if `text` is **not** a valid integer (catch `ValueError`).
- Also handles the case where `text` is `None` or another non-string type (catch `TypeError`), returning `None`.

Handle both error types with a **single** `except` line using a tuple.


In [ ]:
def parse_int(text):
    # YOUR CODE HERE
    try:
      return int(text)
    except (ValueError, TypeError):
      return None


# --- Test cases (do not modify) ---
print(parse_int("42"))     # Expected: 42
print(parse_int("-7"))     # Expected: -7
print(parse_int("3.14"))   # Expected: None   (not a valid int)
print(parse_int("hello"))  # Expected: None
print(parse_int(None))     # Expected: None


42
-7
None
None
None


## Assignment 3 — Inventory lookup with `finally`  ⭐⭐⭐ *Intermediate (20 pts)*

You're given an inventory dictionary. Write `get_quantity(inventory, item)` that:
- Returns the quantity for `item`.
- If `item` is missing, catch `KeyError` and return `0`.
- **Always** prints `f"Lookup complete for: {item}"` using a `finally` block — whether the item was found or not.

Make sure `finally` runs in **both** the found and not-found cases.


In [ ]:
def get_quantity(inventory, item):
    # YOUR CODE HERE
    try:
      return inventory[item]
    except KeyError:
      return 0
    finally:
      print(f"Lookup complete for: {item}")



# --- Test cases (do not modify) ---
stock = {"apples": 12, "bananas": 5}

print(get_quantity(stock, "apples"))

# Expected output:
#   Lookup complete for: apples
#   12
#
print(get_quantity(stock, "cherries"))
# Expected output:
#   Lookup complete for: cherries
#   0


Lookup complete for: apples
12
Lookup complete for: cherries
0


## Assignment 4 — Custom `PasswordError`  ⭐⭐⭐⭐ *Advanced (25 pts)*

Create a **custom exception** called `PasswordError` (inheriting from `Exception`).

Then write `validate_password(password)` that **raises** `PasswordError` with an appropriate message if:
- the password is shorter than 8 characters → `"Password too short"`
- the password has no digit → `"Password needs a number"`

If the password is valid, return the string `"Password OK"`.

Finally, write a small loop (already stubbed) that tries each test password and prints either `"Password OK"` or the caught error message.


In [ ]:
# Step 1: define your custom exception
# YOUR CODE HERE

class MyError(Exception):
  pass


def validate_password(password):
    # Step 2: raise PasswordError for the two rules above, else return "Password OK"
    # YOUR CODE HERE
    pass


# --- Test cases (do not modify) ---
# for pw in ["abc", "abcdefgh", "abcd1234"]:
#     try:
#         print(validate_password(pw))
#     except PasswordError as e:
#         print(f"Invalid: {e}")
#
# Expected output:
#   Invalid: Password too short
#   Invalid: Password needs a number
#   Password OK


## Assignment 5 — Mini banking system  ⭐⭐⭐⭐⭐ *Challenge (30 pts)*

Build a small `BankAccount` class that uses a **custom exception hierarchy** and every tool from this notebook.

**Requirements:**

1. Define a base exception `BankError(Exception)`.
2. Define two subclasses: `InsufficientFundsError(BankError)` and `InvalidAmountError(BankError)`.
3. `BankAccount.__init__(self, balance=0)` stores the starting balance.
4. `deposit(self, amount)`:
   - Raise `InvalidAmountError` if `amount <= 0`.
   - Otherwise add to the balance and return the new balance.
5. `withdraw(self, amount)`:
   - Raise `InvalidAmountError` if `amount <= 0`.
   - Raise `InsufficientFundsError` if `amount > balance` (give it a `.shortfall` attribute equal to `amount - balance`).
   - Otherwise subtract and return the new balance.
6. The stubbed test loop should catch `BankError` (the **base** class) so a single `except` handles both error types, and print the message plus, when present, the shortfall.


In [ ]:
# Step 1: build the exception hierarchy
# YOUR CODE HERE


# Step 2: build the BankAccount class
class BankAccount:
    def __init__(self, balance=0):
        # YOUR CODE HERE
        pass

    def deposit(self, amount):
        # YOUR CODE HERE
        pass

    def withdraw(self, amount):
        # YOUR CODE HERE
        pass


# --- Test cases (do not modify) ---
# account = BankAccount(100)
# operations = [
#     ("deposit", 50),     # ok  -> 150
#     ("withdraw", 30),    # ok  -> 120
#     ("withdraw", 500),   # InsufficientFundsError (shortfall 380)
#     ("deposit", -10),    # InvalidAmountError
# ]
#
# for action, amount in operations:
#     try:
#         if action == "deposit":
#             new_balance = account.deposit(amount)
#         else:
#             new_balance = account.withdraw(amount)
#         print(f"{action}({amount}) OK -> balance {new_balance}")
#     except BankError as e:
#         msg = f"{action}({amount}) FAILED: {e}"
#         if hasattr(e, "shortfall"):
#             msg += f" (short by {e.shortfall})"
#         print(msg)
#
# Expected output:
#   deposit(50) OK -> balance 150
#   withdraw(30) OK -> balance 120
#   withdraw(500) FAILED: <your message>  (short by 380)
#   deposit(-10) FAILED: <your message>
